In [1]:
%pip install gymnasium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import gymnasium as gym

# on crée l'environnement frozenlake unwrapped pour avoir accès à env.P
env = gym.make('FrozenLake-v1').unwrapped
env.reset()

(0, {'prob': 1})

# TP2
### SANNA Thomas, M1 Data Engineer

## Exercice 1. Value itération
### 1.1. Fonction Value_iteration

In [3]:
def value_iteration(env, gamma=1.0, threshold=1e-20, iterations=1000):
    # On initialise la table des valeurs à zéro pour tous les états
    value_table = np.zeros(env.observation_space.n)
    
    # On itère jusqu'à convergence ou jusqu'au nombre max d'itérations
    for i in range(iterations):
        # On sauvegarde la value_table de l'itération précédente
        # car on va l'utiliser pour calculer les nouvelles valeurs
        prev_value_table = np.copy(value_table)
        
        # Pour chaque état, on calcule sa nouvelle valeur
        for state in range(env.observation_space.n):
            # On va calculer Q(s,a) pour toutes les actions possibles
            q_values = []
            
            # Pour chaque action possible dans cet état
            for action in range(env.action_space.n):
                # On calcule Q(s,a) = somme sur s' de P(s'|s,a) * [R(s,a,s') + gamma * V(s')]
                q_value = 0
                # env.P[state][action] contient les transitions possibles
                # Pour chaque transition possible (prob, next_state, reward, done)
                for prob, next_state, reward, done in env.P[state][action]:
                    # On ajoute la contribution de cette transition
                    q_value += prob * (reward + gamma * prev_value_table[next_state])
                q_values.append(q_value)
            
            # V*(s) = max sur toutes les actions de Q*(s,a)
            value_table[state] = max(q_values)
        
        # On vérifie la convergence : si la différence entre l'itération actuelle 
        # et la précédente est très petite, on peut s'arrêter
        if np.max(np.abs(value_table - prev_value_table)) <= threshold:
            print(f'Value iteration a convergé après {i+1} itérations')
            break
    
    return value_table

### 1.2. Extraction de la politique optimale à partir de la fonction de valeur optimale calculée en 1.1.


In [4]:
def extract_policy(value_table, env, gamma=1.0):
    # On initialise la politique avec des zéros (action 0 pour tous les états)
    policy = np.zeros(env.observation_space.n)
    
    # Pour chaque état, on va trouver la meilleure action
    for state in range(env.observation_space.n):
        # On calcule Q(s,a) pour toutes les actions
        q_values = []
        
        for action in range(env.action_space.n):
            # Q(s,a) = somme sur s' de P(s'|s,a) * [R(s,a,s') + gamma * V(s')]
            q_value = 0
            for prob, next_state, reward, done in env.P[state][action]:
                q_value += prob * (reward + gamma * value_table[next_state])
            q_values.append(q_value)
        
        # La politique optimale choisit l'action qui maximise Q(s,a)
        # argmax renvoie l'index de la valeur maximale, donc l'action optimale
        policy[state] = np.argmax(q_values)
    
    return policy

### 1.3. Résolution du Frozen Lake en intégration les deux fonctions précédentes.

In [5]:
# Etape 1 : On calcule la fonction de valeur optimale
optimal_value_function = value_iteration(env)

print("Fonction de valeur optimale :")
print(optimal_value_function.reshape(4,4))
print()

# Etape 2 : On extrait la politique optimale de cette fonction de valeur
optimal_policy = extract_policy(optimal_value_function, env)

print("Politique optimale (Value Iteration) :")
print(optimal_policy.reshape(4,4))
print()

# Affichage plus lisible : 0=Gauche, 1=Bas, 2=Droite, 3=Haut
actions_map = {0: '←', 1: '↓', 2: '→', 3: '↑'}
print("Politique optimale avec directions :")
for i in range(4):
    for j in range(4):
        action = int(optimal_policy[i*4 + j])
        print(actions_map[action], end=' ')
    print()

Fonction de valeur optimale :
[[0.82352941 0.82352941 0.82352941 0.82352941]
 [0.82352941 0.         0.52941176 0.        ]
 [0.82352941 0.82352941 0.76470588 0.        ]
 [0.         0.88235294 0.94117647 0.        ]]

Politique optimale (Value Iteration) :
[[0. 3. 3. 3.]
 [0. 0. 0. 0.]
 [3. 1. 0. 0.]
 [0. 2. 1. 0.]]

Politique optimale avec directions :
← ↑ ↑ ↑ 
← ← ← ← 
↑ ↓ ← ← 
← → ↓ ← 


## 2. Policy iteration
### 2.1. Calculer la value fonction à l'aide de la politique

In [6]:
def compute_value_function(policy, env, gamma=1.0, threshold=1e-20, iterations=1000):
    # On initialise la value_table à zéro
    value_table = np.zeros(env.observation_space.n)
    
    for i in range(iterations):
        # On garde une copie des valeurs de l'itération précédente
        prev_value_table = np.copy(value_table)
        
        # Pour chaque état
        for state in range(env.observation_space.n):
            # On sélectionne l'action dictée par la politique
            action = int(policy[state])
            
            # On calcule V^pi(s) = somme sur s' de P(s'|s,a) * [R(s,a,s') + gamma * V^pi(s')]
            # avec a = politique(s)
            q_value = 0
            for prob, next_state, reward, done in env.P[state][action]:
                q_value += prob * (reward + gamma * prev_value_table[next_state])
            
            # On met à jour la valeur de l'état
            value_table[state] = q_value
        
        # On vérifie la convergence
        if np.max(np.abs(value_table - prev_value_table)) <= threshold:
            break
    
    return value_table

In [7]:
# On a déjà défini extract_policy pour l'exercice 1, 
# on va la réutiliser ici car c'est exactement la même chose !
# extract_policy prend une value function et renvoie la politique qui maximise les Q values

print("La fonction extract_policy a déjà été définie dans l'exercice 1.2")

La fonction extract_policy a déjà été définie dans l'exercice 1.2


### 2.2. Extraction de la politique à partir de la value fonction

In [11]:
# Cette fonction est identique à celle définie en 1.2
# Elle prend une value function et renvoie la politique qui maximise les Q-values
# Exemple d'utilisation :

# On créé une politique de test (toutes les actions à 0)
test_policy = np.zeros(env.observation_space.n)

# On calcule la value function associée à cette politique
test_value_function = compute_value_function(test_policy, env)

print("Value function calculée avec la politique de test :")
print(test_value_function.reshape(4,4))
print()

# On extrait une meilleure politique à partir de cette value function
improved_policy = extract_policy(test_value_function, env)

print("Politique améliorée extraite :")
print(improved_policy.reshape(4,4))

Value function calculée avec la politique de test :
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

Politique améliorée extraite :
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 1. 0.]]


### 2.3. Intégration des deux étapes précédentes

In [9]:
def policy_iteration(env, gamma=1.0, iterations=1000):
    # On initialise une politique aléatoire (ici tout à zéro)
    policy = np.zeros(env.observation_space.n)
    
    for i in range(iterations):
        # Etape 1 : On calcule la value function avec la politique actuelle
        value_function = compute_value_function(policy, env, gamma)
        
        # Etape 2 : On extrait une nouvelle politique à partir de cette value function
        new_policy = extract_policy(value_function, env, gamma)
        
        # Si la nouvelle politique est identique à l'ancienne, c'est qu'on a convergé !
        # La politique optimale a été trouvée
        if np.array_equal(policy, new_policy):
            print(f'Policy iteration a convergé après {i+1} itérations')
            break
        
        # Sinon on continue avec la nouvelle politique
        policy = new_policy
    
    return policy

In [10]:
# On applique Policy Iteration sur l'environnement Frozen Lake
optimal_policy_pi = policy_iteration(env)

print("\nPolitique optimale (Policy Iteration) :")
print(optimal_policy_pi.reshape(4,4))
print()

# Affichage avec les directions pour mieux comprendre
actions_map = {0: '←', 1: '↓', 2: '→', 3: '↑'}
print("Politique optimale avec directions :")
for i in range(4):
    for j in range(4):
        action = int(optimal_policy_pi[i*4 + j])
        print(actions_map[action], end=' ')
    print()

Policy iteration a convergé après 7 itérations

Politique optimale (Policy Iteration) :
[[0. 3. 3. 3.]
 [0. 0. 0. 0.]
 [3. 1. 0. 0.]
 [0. 2. 1. 0.]]

Politique optimale avec directions :
← ↑ ↑ ↑ 
← ← ← ← 
↑ ↓ ← ← 
← → ↓ ← 
